In [1]:
# import libraries 
!pip install selenium
!pip install webdriver-manager
!pip install chromedriver-autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import chromedriver_autoinstaller
import requests
import time
import datetime
import smtplib

In [2]:
# --- CONFIG ---
URL = "https://www.amazon.com/Data-Science-Analytics-Funny-T-Shirt/dp/B07PB9XYH8/ref=sr_1_4?crid=1F9HL5DXGAHGM&dib=eyJ2IjoiMSJ9.lQ2zbcZw8CNAgXEQY1xMfLmSoLgnN3-BJLQLGTkwS5ICsuizq8MRUavR8w0V2fpGiv0gLUwpfOE1H55LrAwfmKMba2VJvY8Ez7Fyi99b0p-MkF0L6mckaxr0fgpqpPaZW8B2dSYCdIrx21CcCrBVOCFOKOmXcXeOV-fLX3n3q3o7RxT4Fwkz9HHA47MeH4jMmwR1KSBGAZN6Y1p7tCPr6naLeyBHa-Dz-BNzONNLrfLCYH1EWfZdaOgIufHNDLJo-RhSl9CVUGuDzvjaG7NNy5pUU4PkMnGxoAWVp7mfoEY.fLaAcbS-TAZDZbfTGrAe6uJK84zfhOEXkv8UxJaFhQ0&dib_tag=se&keywords=funny%2Bgot%2Bdata%2Btshirt&qid=1778012237&sprefix=funny%2Bgot%2Bdata%2Btshir%2Caps%2C285&sr=8-4&customId=B0752XJYNL&customizationToken=MC_Assembly_1%23B0752XJYNL&th=1&psc=1"

# --- INSTALL CORRECT CHROMEDRIVER ---
import chromedriver_autoinstaller
chromedriver_autoinstaller.install()  # Automatically installs correct ChromeDriver

# --- SETUP DRIVER ---
from selenium import webdriver
from bs4 import BeautifulSoup
import time

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
# options.add_argument("--headless")  # optional

driver = webdriver.Chrome(options=options)

# --- LOAD PAGE ---
driver.get(URL)
time.sleep(5)  # wait for JS to fully load

# --- PARSE PAGE ---
soup = BeautifulSoup(driver.page_source, "html.parser")

# --- GET TITLE ---
title_tag = soup.find(id="productTitle")
title = title_tag.get_text(strip=True) if title_tag else "Title not found"

# --- GET PRICE (robust approach) ---
price_tag = soup.find(id="priceblock_ourprice") or soup.find(id="priceblock_dealprice")
if not price_tag:
    price_tags = soup.find_all("span", class_="a-offscreen")
    price_tag = next((t for t in price_tags if t.get_text(strip=True).startswith("$")), None)

price_text = price_tag.get_text(strip=True) if price_tag else "Price not found"

print("Product Title:", title)
print("Product Price:", price_text)

# --- CLEANUP ---
driver.quit()

Product Title: Data Science Analytics Funny Got Data? T-Shirt T-Shirt
Product Price: $20.99


In [3]:
# Create a Timestamp for your output to track when data was collected

import datetime

today = datetime.date.today()

print(today)

2026-05-06


In [4]:
# Create CSV and write headers and data into the file

import csv 

header = ['Title', 'Price', 'Date']
data = [title, price_text, today]


with open('AmazonWebScraperDataset.csv', 'w', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerow(data)

In [5]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Tina McKenney\Documents\Python\AmazonWebScraperDataset.csv')

print(df)

                                               Title   Price        Date
0  Data Science Analytics Funny Got Data? T-Shirt...  $20.99  2026-05-06


In [6]:
#Now we are appending data to the csv

with open('AmazonWebScraperDataset.csv', 'a+', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow(data)

In [7]:
#Combine all of the above code into one function

def check_price():
    from selenium import webdriver
    from bs4 import BeautifulSoup
    import time, datetime, csv

    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # optional

    driver = webdriver.Chrome(options=options)
    driver.get(URL)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    title_tag = soup.find(id="productTitle")
    title = title_tag.get_text(strip=True) if title_tag else "Title not found"

    price_tag = soup.find(id="priceblock_ourprice") or soup.find(id="priceblock_dealprice")
    if not price_tag:
        price_tags = soup.find_all("span", class_="a-offscreen")
        price_tag = next((t for t in price_tags if t.get_text(strip=True).startswith("$")), None)

    price = price_tag.get_text(strip=True) if price_tag else "Price not found"

    today = datetime.date.today()

    data = [title, price, today]

    with open('AmazonWebScraperDataset.csv', 'a+', newline='', encoding='UTF8') as f:
        writer = csv.writer(f)
        writer.writerow(data)

    print("Logged:", data)

    driver.quit()

In [ ]:
# Runs check_price after a set time and inputs data into your CSV

while(True):
    check_price()
    time.sleep(86400)

Logged: ['Data Science Analytics Funny Got Data? T-Shirt T-Shirt', '$20.99', datetime.date(2026, 5, 6)]


In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\Tina McKenney\Documents\Python\AmazonWebScraperDataset.csv')

print(df)

In [ ]:
# If uou want to try sending yourself an email (just for fun) when a price hits below a certain level you can try it
# out with this script

def send_mail():
    server = smtplib.SMTP_SSL('smtp.gmail.com',465)
    server.ehlo()
    #server.starttls()
    server.ehlo()
    server.login('tgibbs0318@gmail.com','xxxxxxxxxxxxxx')
    
    subject = "The Shirt you want is below $15! Now is your chance to buy!"
    body = "Tina, This is the moment we have been waiting for. Now is your chance to pick up the shirt of your dreams. Don't mess it up! Link here: https://www.amazon.com/Data-Science-Analytics-Funny-T-Shirt/dp/B07PB9XYH8/ref=sr_1_4?crid=1F9HL5DXGAHGM&dib=eyJ2IjoiMSJ9.lQ2zbcZw8CNAgXEQY1xMfLmSoLgnN3-BJLQLGTkwS5ICsuizq8MRUavR8w0V2fpGiv0gLUwpfOE1H55LrAwfmKMba2VJvY8Ez7Fyi99b0p-MkF0L6mckaxr0fgpqpPaZW8B2dSYCdIrx21CcCrBVOCFOKOmXcXeOV-fLX3n3q3o7RxT4Fwkz9HHA47MeH4jMmwR1KSBGAZN6Y1p7tCPr6naLeyBHa-Dz-BNzONNLrfLCYH1EWfZdaOgIufHNDLJo-RhSl9CVUGuDzvjaG7NNy5pUU4PkMnGxoAWVp7mfoEY.fLaAcbS-TAZDZbfTGrAe6uJK84zfhOEXkv8UxJaFhQ0&dib_tag=se&keywords=funny%2Bgot%2Bdata%2Btshirt&qid=1778012237&sprefix=funny%2Bgot%2Bdata%2Btshir%2Caps%2C285&sr=8-4&customId=B0752XJYNL&customizationToken=MC_Assembly_1%23B0752XJYNL&th=1&psc=1"
   
    msg = f"Subject: {subject}\n\n{body}"
    
    server.sendmail(
        'tgibbs0318@gmail.com',
        msg
     
    )